# LLM Firewall - Stage 4: Evaluation & Metrics

This notebook performs a systematic evaluation of the fine-tuned LLM Firewall (**v2**) and compares it to a zero-shot **GPT-4o-mini** baseline.

## 1. Setup & Data Loading

In [ ]:
import json
import pandas as pd
import numpy as np
import torch
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, roc_curve, auc
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel

def load_and_split_data(benign_path, malicious_path, test_size=0.2):
    """Loads datasets and creates a stratified test split."""
    print(f"Loading data from {benign_path} and {malicious_path}")
    with open(benign_path, 'r') as f:
        benign = json.load(f)
    with open(malicious_path, 'r') as f:
        malicious = json.load(f)

    df_benign = pd.DataFrame(benign)
    df_malicious = pd.DataFrame(malicious)

    df_combined = pd.concat([df_benign, df_malicious], ignore_index=True)
    print(f"Total combined dataset size: {len(df_combined)}")

    # Stratified split
    train_df, test_df = train_test_split(
        df_combined,
        test_size=test_size,
        stratify=df_combined['label'],
        random_state=42
    )

    print(f"Test set size: {len(test_df)} (Benign: {len(test_df[test_df['label']==0])}, Malicious: {len(test_df[test_df['label']==1])})")
    return test_df

In [8]:
# Path to your data files
benign_path = '/content/benign.json'
malicious_path = '/content/up_mal.json'

test_df = load_and_split_data(benign_path, malicious_path)

Loading data from /content/benign.json and /content/up_mal.json
Total combined dataset size: 14759
Test set size: 2952 (Benign: 1530, Malicious: 1422)


## 2. Defender Evaluation (BERT v2)

We use the fine-tuned BERT model (which has been hardened via the adversarial loop).

In [9]:
def evaluate_bert(test_df, model_id='kunjcr2/bert-lora', device='cuda'):
    """Runs inference and returns raw probabilities and predictions."""
    print(f"Loading model: {model_id} onto {device}")
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    base_model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased", num_labels=2)
    model = PeftModel.from_pretrained(base_model, model_id)
    model = model.to(device).eval()

    texts = test_df['user'].tolist()
    labels = test_df['label'].tolist()

    probs = []
    preds = []

    print("Starting BERT inference...")
    batch_size = 32
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i : i + batch_size]
        enc = tokenizer(batch_texts, padding=True, truncation=True, max_length=128, return_tensors="pt").to(device)

        with torch.no_grad():
            logits = model(**enc).logits
            softmax_probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            batch_preds = logits.argmax(-1).cpu().numpy()

        probs.extend(softmax_probs)
        preds.extend(batch_preds)

        if (i // batch_size) % 10 == 0:
            print(f"Processed {i + len(batch_texts)} / {len(texts)} samples")

    accuracy = accuracy_score(labels, preds)
    print(f"\nOverall Accuracy: {accuracy:.4f}")

    return np.array(probs), np.array(preds), labels

bert_probs, bert_preds, true_labels = evaluate_bert(test_df)

Loading model: kunjcr2/bert-lora onto cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Starting BERT inference...
Processed 32 / 2952 samples
Processed 352 / 2952 samples
Processed 672 / 2952 samples
Processed 992 / 2952 samples
Processed 1312 / 2952 samples
Processed 1632 / 2952 samples
Processed 1952 / 2952 samples
Processed 2272 / 2952 samples
Processed 2592 / 2952 samples
Processed 2912 / 2952 samples

Overall Accuracy: 0.9624


## 3. Threshold Optimization (Fixed-FPR at 1%)

In security contexts, a False Positive (blocking a legitimate user) is often more destructive than a False Negative (letting an attack through). We optimize the threshold to maintain a strict **1% FPR**.

In [10]:
def analyze_fixed_fpr(probs, labels, target_fpr=0.01):
    """Finds the detection rate (Recall) at a specific False Positive Rate."""
    fprs, tprs, thresholds = roc_curve(labels, probs)
    idx = np.abs(fprs - target_fpr).argmin()

    optimal_threshold = thresholds[idx]
    recall_at_fpr = tprs[idx]

    print(f"Results for fixed FPR of {target_fpr * 100}%:")
    print(f" - Optimal Threshold: {optimal_threshold:.4f}")
    print(f" - Recall (Detection Rate): {recall_at_fpr * 100:.2f}%")

    return optimal_threshold, recall_at_fpr

opt_threshold, recall_at_1fpr = analyze_fixed_fpr(bert_probs, true_labels)

Results for fixed FPR of 1.0%:
 - Optimal Threshold: 0.9998
 - Recall (Detection Rate): 99.30%


## 4. Latency Benchmark

Measure inference throughput for the BERT classifier.

In [12]:
def benchmark_latency(texts, model_id='kunjcr2/bert-lora', device='cuda'):
    # Loading model already done in Section 2, but for clarity:
    print("Starting latency benchmark...")
    # Use a small subset
    test_texts = texts[:100]

    # Tokenizer warmup
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    base_model = AutoModelForSequenceClassification.from_pretrained("google-bert/bert-base-uncased", num_labels=2)
    model = PeftModel.from_pretrained(base_model, model_id).to(device).eval()

    start_time = time.time()
    for text in test_texts:
        enc = tokenizer(text, truncation=True, max_length=512, return_tensors="pt").to(device)
        with torch.no_grad():
            model(**enc)

    total_time = time.time() - start_time
    avg_time = (total_time / 100) * 1000
    print("-"*100)
    print(f"Average Latency: {avg_time:.2f} ms per prompt")
    return avg_time

avg_latency = benchmark_latency(test_df['user'].tolist())

Starting latency benchmark...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


----------------------------------------------------------------------------------------------------
Average Latency: 19.79 ms per prompt
